DATASET GENERATOR

In [1]:
import pandas as pd

In [2]:
# 1. Caricamento dei file
NORMALIZED_AUDIO=False
normalized_str="[NORMALIZED_AUDIO]" * NORMALIZED_AUDIO
NOISE_REDUCTION=False
noise_reduction_str="[NOISE_REDUCTION]" * NOISE_REDUCTION
HIGH_PASS_FILTER=False
high_pass_filter_str="[HIGH_PASS_FILTER]" * HIGH_PASS_FILTER

df_metadati = pd.read_csv(f'{normalized_str}{noise_reduction_str}{high_pass_filter_str}audio_samples_filtered_and_preprocessed_metadata.csv')
df_features = pd.read_csv(f'{normalized_str}{noise_reduction_str}{high_pass_filter_str}audio_samples_filtered_and_preprocessed_features.csv')
df_portate = pd.read_csv('ValoriPortata.csv', sep=';')
CSV_OUTPUT_PATH = f"{normalized_str}{noise_reduction_str}{high_pass_filter_str}audio_dataset.csv"

In [3]:
df_portate['portata'] = df_portate['portata'] / 900     # 15 minuti = 900 secondi
df_portate['portata'] = df_portate['portata'] / 4       # perchè la portata è quella totale, distribuitia uniformemente sui 4 GRF

# 2. Conversione dei timestamp in oggetti datetime
# Assicurati che il formato sia corretto per entrambi i dataframe
df_metadati['timestamp'] = pd.to_datetime(df_metadati['timestamp'])
#print(df_metadati["timestamp"].head(1))
df_metadati['timestamp'] = df_metadati['timestamp'].dt.tz_convert('Europe/Rome')
#print(df_metadati["timestamp"].head(1))
df_portate['timestamp'] = pd.to_datetime(df_portate['timestamp'])
#print(df_portate["timestamp"].head(1))
df_portate['timestamp'] = df_portate['timestamp'].dt.tz_localize('Europe/Rome',nonexistent='shift_forward', ambiguous='infer')
#print(df_portate["timestamp"].head(1))

cols_to_drop = [c for c in df_metadati.columns if c != 'filename']

# 3. Unione di metadati e feature (basata sul filename)
df_audio = pd.merge(df_metadati, df_features, on='filename')

# 4. Ordinamento per timestamp
# Fondamentale per far funzionare merge_asof
df_audio = df_audio.sort_values('timestamp')
df_portate = df_portate.sort_values('timestamp')

# 5. Merge temporale "approssimativo"
# Questo comando cerca per ogni riga di df_audio il valore di df_portate 
# con il timestamp più vicino.
df_final = pd.merge_asof(
    df_audio, 
    df_portate, 
    on='timestamp', 
    direction='forward'
)

df_final = df_final.drop(columns=cols_to_drop)
df_final.set_index('filename', inplace=True)

In [4]:
# Visualizzazione del risultato
print(df_final.head())

                                ae_mean    ae_std    ae_min    ae_max  \
filename                                                                
audio_2026-02-23T09-26-56.wav  0.000627  0.000070  0.000494  0.000725   
audio_2026-02-23T09-26-57.wav  0.000577  0.000073  0.000352  0.000678   
audio_2026-02-23T09-26-58.wav  0.000643  0.000066  0.000567  0.000768   
audio_2026-02-23T09-26-59.wav  0.000669  0.000093  0.000445  0.000831   
audio_2026-02-23T09-28-08.wav  0.000650  0.000059  0.000560  0.000741   

                                  ae_q1  ae_median     ae_q3    ae_iqr  \
filename                                                                 
audio_2026-02-23T09-26-56.wav  0.000553   0.000608  0.000689  0.000136   
audio_2026-02-23T09-26-57.wav  0.000523   0.000576  0.000635  0.000112   
audio_2026-02-23T09-26-58.wav  0.000584   0.000641  0.000693  0.000110   
audio_2026-02-23T09-26-59.wav  0.000621   0.000658  0.000748  0.000126   
audio_2026-02-23T09-28-08.wav  0.000597   0.

In [5]:
# Salvataggio
df_final.to_csv(CSV_OUTPUT_PATH, index=True)

In [6]:
df_prova = df_final.reset_index()[['filename', 'portata']]

# 2. Salvataggio del dataframe di prova in un file CSV
# Usiamo index=False per evitare che pandas aggiunga una colonna di indici numerici (0, 1, 2...) nel file
df_prova.to_csv(f'{normalized_str}{noise_reduction_str}{high_pass_filter_str}dataset_assegnazione_portate.csv', index=False)

C:\Users\chris\AppData\Local\Temp\ipykernel_14144\1849764171.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_prova = df_final.reset_index()[['filename', 'portata']]
